# CPRI Hackathon — Person 1 Stage 7: ML Classification & Anomaly Models

This notebook demonstrates Person 1 Stage 7 machine learning classification & anomaly models for Task 01:
1. **Dataset Loading & Feature Set Preparation** (Sets A, B, C, D, E).
2. **5-Fold Stratified Cross-Validation Evaluation** across 15 candidate configurations.
3. **Out-of-Fold (OOF) Decision Threshold Optimization**.
4. **Permutation Feature Importance** analysis.
5. **False Positive & False Negative Error Analysis**.
6. **Final Inference on Test_Data** (350 records).

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath('..'))

from src.dataset_loader import load_training_data, load_test_data, TASK01_TARGET
from src.stage5_features import create_stage5_features
from src.stage7_ml_models import (
    evaluate_all_stage7_models,
    optimize_decision_threshold,
    prepare_feature_sets,
    get_model_pipelines,
    calculate_permutation_importance
)

sns.set_theme(style='whitegrid', palette='muted')
print('Setup complete.')

## 1. Load Datasets & Generate Stage 5 Features

In [ ]:
df_train = load_training_data()
df_test = load_test_data()

df_train_feat, normal_models = create_stage5_features(df_train, fit_normal_models=True)
df_test_feat, _ = create_stage5_features(df_test, fit_normal_models=False, normal_models=normal_models)
print(f'Training feature matrix shape: {df_train_feat.shape} | Test feature matrix shape: {df_test_feat.shape}')

## 2. Evaluate 15 Candidate Models x Feature Sets (5-Fold Stratified CV)

In [ ]:
comp_df, all_oof_probs, best_info = evaluate_all_stage7_models(df_train_feat)
display(comp_df.head(10))

## 3. Decision Threshold Optimization

In [ ]:
y_true = (df_train[TASK01_TARGET] == 'Invalid').astype(int).values
best_probs = all_oof_probs[best_info['config_key']]
thresh_df = optimize_decision_threshold(y_true, best_probs)
display(thresh_df.head(10))

## 4. Feature Importance & Final Inference Demonstration

In [ ]:
train_sets, test_sets, names_dict = prepare_feature_sets(df_train_feat, df_test_feat)
best_fs = best_info['feature_set_name']
best_model_name = best_info['model_name']
pipeline = get_model_pipelines()[best_model_name]

imp_df = calculate_permutation_importance(pipeline, train_sets[best_fs], y_true, names_dict[best_fs])
display(imp_df.head(15))

# Final inference on 350 Test records
test_probs = pipeline.predict_proba(test_sets[best_fs].values)[:, 1]
best_thresh = thresh_df.iloc[0]['Threshold']
test_preds = np.where(test_probs >= best_thresh, 'Invalid', 'Valid')
print(f'Test_Data invalid count: {(test_preds == "Invalid").sum()} / 350')